# 🧬 DNA 16S rRNA Species Classifier — Fine-Tuning Notebook

## Objectives
1. Download **real NCBI 16S ribosomal RNA** sequences from the BLAST FTP database.
2. Resolve full NCBI taxonomy (kingdom → species) for every sequence.
3. Build a **balanced, species-level** labeled dataset (~50k sequences).
4. Fine-tune a HuggingFace DNA language model for multi-class species classification.
5. Export a **plug-and-play** model directory that drops straight into `edna_pipeline/models/dnabert2_16s_species`.

## Expected Runtime
| Stage | Approx. Time (P100/T4) |
|---|---|
| Install + PyTorch fix | 3–5 min |
| Download + extract NCBI data | 5–10 min |
| Taxonomy resolution + dataset build | 3–5 min |
| Tokenization | 2–5 min |
| Fine-tuning (3 epochs, ~50k seqs) | 30–60 min |
| Evaluation + export | 2–3 min |
| **Total** | **~45–90 min** |

## Target project layout
```
/home/megh/edna/
  edna_pipeline/
    models/
      dnabert2_16s_species/   ← unzip artifact here
        config.json
        pytorch_model.bin  (or model.safetensors)
        tokenizer_config.json
        special_tokens_map.json
        tokenizer.json  (or vocab.txt)
        label_encoder.pkl
        model_info.json
```

## Future database note
For broader biodiversity coverage in future runs, also consider:
- `18S_fungal_sequences`
- `28S_fungal_sequences`
- `ITS_eukaryote_sequences`
- `ITS_RefSeq_Fungi`

Available at https://ftp.ncbi.nlm.nih.gov/blast/db/

## P100 / Kaggle Compatibility
This notebook explicitly handles the **Pascal GPU (sm_60) CUDA compatibility issue**.
PyTorch builds with CUDA >= 12.8 dropped Pascal support. We install PyTorch with CUDA 12.6
to ensure P100 kernels are available.

---
## 1 · Install Dependencies + Fix P100 CUDA Compatibility

In [ ]:
%%bash
set -e

echo '=== Step 1a: Install apt packages ==='
apt-get update -qq && apt-get install -y -qq ncbi-blast+ pigz > /dev/null 2>&1
echo "blastdbcmd: $(which blastdbcmd)"
blastdbcmd -version

echo ''
echo '=== Step 1b: Install PyTorch with CUDA 12.6 (P100 Pascal sm_60 support) ==='
echo 'PyTorch CUDA>=12.8 dropped Pascal support — we force cu126.'
pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

echo ''
echo '=== Step 1c: Install Python packages ==='
pip install -q \
    transformers>=4.38 \
    datasets>=2.18 \
    accelerate>=0.27 \
    evaluate \
    scikit-learn>=1.3 \
    biopython>=1.83 \
    safetensors \
    sentencepiece \
    protobuf

echo ''
echo '>>> All installs complete.'

---
## 1b · Verify GPU + CUDA + PyTorch Compatibility

In [ ]:
import torch
print(f'PyTorch version:  {torch.__version__}')
print(f'CUDA available:   {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA version:     {torch.version.cuda}')
    print(f'GPU:              {torch.cuda.get_device_name(0)}')
    cap = torch.cuda.get_device_capability(0)
    print(f'Compute cap:      {cap[0]}.{cap[1]}')
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU Memory:       {mem_gb:.1f} GB')
    
    # Smoke test: do a simple CUDA operation to catch kernel issues early
    try:
        x = torch.randn(100, 100, device='cuda')
        y = torch.matmul(x, x)
        del x, y
        torch.cuda.empty_cache()
        print('CUDA smoke test:  ✅ PASSED')
    except RuntimeError as e:
        print(f'CUDA smoke test:  ❌ FAILED — {e}')
        print('\n*** You may need to restart the runtime after the PyTorch reinstall. ***')
        print('*** Go to Runtime → Restart runtime, then run cells from Cell 2 onward. ***')
        raise
else:
    print('\n⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU')
    print('    This notebook requires a GPU (P100 or T4).')

---
## 2 · Configure Directories & Constants

In [ ]:
import os, pathlib, shutil

# ── Paths ──────────────────────────────────────────────────────────
WORK_DIR   = pathlib.Path('/content/work')
OUTPUT_DIR = pathlib.Path('/content/output/dnabert2_16s_model')
BLAST_DIR  = WORK_DIR / 'blastdb'
TAX_DIR    = WORK_DIR / 'taxdump'
DATA_DIR   = WORK_DIR / 'data'
MODEL_DIR  = OUTPUT_DIR / 'dnabert2_16s_species'

for d in [WORK_DIR, OUTPUT_DIR, BLAST_DIR, TAX_DIR, DATA_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Dataset hyper-parameters ───────────────────────────────────────
MIN_SAMPLES_PER_SPECIES = 5       # minimum samples per species (lowered for 16S DB coverage)
MAX_CLASSES             = 2000    # keep top-N most frequent species
MAX_PER_CLASS           = 200     # cap per-class to reduce imbalance
TARGET_TOTAL            = 50_000  # target dataset size (~)
MIN_SEQ_LEN             = 200     # nt – too-short fragments are noise
MAX_SEQ_LEN             = 2000    # nt – 16S is ~1.5 kb
MAX_TOKEN_LEN           = 512     # transformer context window

# ── Training hyper-parameters ──────────────────────────────────────
EPOCHS          = 3
TRAIN_BATCH     = 16
EVAL_BATCH      = 32
LEARNING_RATE   = 2e-5
WEIGHT_DECAY    = 0.01
WARMUP_STEPS    = 200     # use warmup_steps instead of warmup_ratio (deprecated in v5.2)
FP16            = True

# ── URLs ───────────────────────────────────────────────────────────
BLAST_16S_URL = 'https://ftp.ncbi.nlm.nih.gov/blast/db/16S_ribosomal_RNA.tar.gz'
TAXDUMP_URL   = 'https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz'

print('Configuration OK')
print(f'  WORK_DIR   = {WORK_DIR}')
print(f'  OUTPUT_DIR = {OUTPUT_DIR}')
print(f'  MODEL_DIR  = {MODEL_DIR}')

---
## 3 · Download & Extract NCBI 16S BLAST DB + Taxdump

In [ ]:
import subprocess, sys, urllib.request, time

def run(cmd, **kw):
    """Run shell command with error checking."""
    print(f'  $ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print(f'STDOUT:\n{r.stdout[-2000:] if r.stdout else ""}')
        print(f'STDERR:\n{r.stderr[-2000:] if r.stderr else ""}')
        raise RuntimeError(f'Command failed (rc={r.returncode}): {cmd}')
    return r

def download(url, dest_dir):
    fname = url.rsplit('/', 1)[-1]
    dest = dest_dir / fname
    if dest.exists():
        print(f'  [skip] {dest} already exists ({dest.stat().st_size / 1e6:.1f} MB)')
        return dest
    print(f'  Downloading {url} ...')
    t0 = time.time()
    urllib.request.urlretrieve(url, str(dest))
    elapsed = time.time() - t0
    print(f'  Saved {dest}  ({dest.stat().st_size / 1e6:.1f} MB, {elapsed:.0f}s)')
    return dest

# ── Download 16S BLAST DB ──────────────────────────────────────────
print('=== 16S ribosomal RNA BLAST DB ===')
blast_tar = download(BLAST_16S_URL, WORK_DIR)
run(f'tar -xzf {blast_tar} -C {BLAST_DIR}')
db_files = list(BLAST_DIR.iterdir())
print(f'  Extracted {len(db_files)} files to {BLAST_DIR}')
for f in sorted(db_files)[:10]:
    print(f'    {f.name}')

# ── Download taxdump ───────────────────────────────────────────────
print('\n=== NCBI Taxdump ===')
tax_tar = download(TAXDUMP_URL, WORK_DIR)
run(f'tar -xzf {tax_tar} -C {TAX_DIR}')

# Verify critical files exist
for f in ['nodes.dmp', 'names.dmp']:
    p = TAX_DIR / f
    assert p.exists(), f'Missing {p}'
    print(f'  ✓ {f}  ({p.stat().st_size / 1e6:.1f} MB)')

print('\n✅ All downloads complete.')

---
## 4 · Extract Accession / TaxID / Title / Sequence with `blastdbcmd`

**CRITICAL**: Uses `%T` for numeric TaxID (NOT `%i` which gives sequence ID).

In [ ]:
import pathlib, re as _re

SEQ_TSV = DATA_DIR / 'sequences_raw.tsv'

# Find the BLAST DB name (without extension)
# Look for common index files
nhr_files = sorted(BLAST_DIR.glob('*.nhr')) + sorted(BLAST_DIR.glob('*.ndb'))
if not nhr_files:
    # Try .nin or .nsq as fallback
    nhr_files = sorted(BLAST_DIR.glob('*.nin')) + sorted(BLAST_DIR.glob('*.nsq'))
assert len(nhr_files) > 0, f'No BLAST DB index files found in {BLAST_DIR}! Files: {list(BLAST_DIR.iterdir())}'

# The DB name is the stem without the extension
db_path = str(nhr_files[0])
# Remove extension like .nhr, .ndb, .nin, .nsq
db_name = _re.sub(r'\.(nhr|ndb|nin|nsq|nsi|nsd|nog|nos|not|ntf|nto)$', '', db_path)
# Handle multi-volume: strip trailing .00, .01, etc.
db_name = _re.sub(r'\.\d+$', '', db_name)
print(f'BLAST DB name: {db_name}')

# Verify DB is valid
info_result = run(f'blastdbcmd -db {db_name} -info')
print(info_result.stdout[:500])

# ── CRITICAL: use %T for taxid, NOT %i ──────────────────────────
# %a = accession, %T = taxid (numeric), %t = title, %s = sequence (no linebreaks)
cmd = (
    f'blastdbcmd -db {db_name} -entry all '
    f'-outfmt "%a\t%T\t%t\t%s" '
    f'-out {SEQ_TSV}'
)
print('\nRunning blastdbcmd (may take a few minutes) ...')
result = run(cmd)

# Validate output
assert SEQ_TSV.exists(), f'{SEQ_TSV} was not created!'
file_size = SEQ_TSV.stat().st_size
assert file_size > 0, 'blastdbcmd produced empty output!'
print(f'  Output file size: {file_size / 1e6:.1f} MB')

line_count = int(run(f'wc -l < {SEQ_TSV}').stdout.strip())
assert line_count > 0, 'blastdbcmd produced empty output!'
print(f'\n✅ Extracted {line_count:,} sequence records.')

# Show first few lines for debugging
print('\nSample lines (first 3):')
sample_output = run(f'head -3 {SEQ_TSV}').stdout
for line in sample_output.strip().split('\n'):
    parts = line.split('\t')
    print(f'  accession={parts[0]}, taxid={parts[1] if len(parts)>1 else "?"}, '
          f'title={parts[2][:60] if len(parts)>2 else "?"}, '
          f'seq_len={len(parts[3]) if len(parts)>3 else "?"}')

---
## 5 · Parse `nodes.dmp` + `names.dmp` → Lineage Resolver

In [ ]:
import collections

print('Parsing nodes.dmp ...')
parent_map = {}   # taxid → parent_taxid
rank_map   = {}   # taxid → rank string
with open(TAX_DIR / 'nodes.dmp') as fh:
    for line in fh:
        parts = line.split('\t|\t')
        tid   = int(parts[0].strip())
        ptid  = int(parts[1].strip())
        rank  = parts[2].strip()
        parent_map[tid] = ptid
        rank_map[tid]   = rank
print(f'  {len(parent_map):,} nodes loaded')

print('Parsing names.dmp ...')
name_map = {}  # taxid → scientific name
with open(TAX_DIR / 'names.dmp') as fh:
    for line in fh:
        parts = line.split('\t|\t')
        if 'scientific name' in parts[-1]:
            tid  = int(parts[0].strip())
            name = parts[1].strip()
            name_map[tid] = name
print(f'  {len(name_map):,} scientific names loaded')

# ── Lineage resolver ───────────────────────────────────────────────
RANKS_OF_INTEREST = ['superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
RANK_ALIASES = {'superkingdom': 'kingdom'}  # rename for output

def resolve_lineage(taxid):
    """Walk up the taxonomy tree and return dict of rank→name."""
    lineage = {}
    cur = taxid
    visited = set()
    while cur and cur not in visited:
        visited.add(cur)
        r = rank_map.get(cur)
        if r in RANKS_OF_INTEREST:
            key = RANK_ALIASES.get(r, r)
            lineage[key] = name_map.get(cur, 'Unknown')
        cur = parent_map.get(cur)
    return lineage

# Quick test
test_lin = resolve_lineage(562)  # E. coli
print(f'\nTest lineage (taxid 562 / E. coli):')
for rank in ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']:
    print(f'  {rank:12s}: {test_lin.get(rank, "—")}')
assert 'species' in test_lin, 'Lineage resolver failed for E. coli!'
assert 'Escherichia coli' in test_lin['species'], f'Expected E. coli, got: {test_lin["species"]}'
print('✅ Lineage resolver OK.')

---
## 6 · Build DataFrame with Taxonomy Labels

In [ ]:
import pandas as pd
import re

print('Loading raw sequences TSV ...')
rows = []
bad = 0
bad_reasons = collections.Counter()
with open(SEQ_TSV) as fh:
    for i, line in enumerate(fh):
        parts = line.rstrip('\n').split('\t', 3)
        if len(parts) < 4:
            bad += 1
            bad_reasons['insufficient_columns'] += 1
            continue
        acc, taxid_str, title, seq = parts
        try:
            taxid = int(taxid_str)
        except ValueError:
            bad += 1
            bad_reasons['invalid_taxid'] += 1
            continue
        if taxid <= 0:
            bad += 1
            bad_reasons['zero_taxid'] += 1
            continue
        if len(seq.strip()) < 50:
            bad += 1
            bad_reasons['very_short_seq'] += 1
            continue
        rows.append({'accession': acc.strip(), 'taxid': taxid, 'title': title.strip(), 'sequence': seq.strip()})

print(f'  Parsed {len(rows):,} valid rows,  {bad:,} skipped.')
if bad > 0:
    print(f'  Skip reasons: {dict(bad_reasons)}')
assert len(rows) > 1000, f'FATAL: Too few sequences parsed ({len(rows)}). BLAST DB extraction likely failed.'

df = pd.DataFrame(rows)
print(f'  DataFrame shape: {df.shape}')
print(f'  Unique taxids: {df["taxid"].nunique():,}')

# ── Resolve taxonomy ───────────────────────────────────────────────
print('\nResolving taxonomy for all sequences (this may take a minute) ...')

# Cache lineage lookups (many sequences share the same taxid)
lineage_cache = {}
lineages = []
for tid in df['taxid']:
    if tid not in lineage_cache:
        lineage_cache[tid] = resolve_lineage(tid)
    lineages.append(lineage_cache[tid])

print(f'  Resolved {len(lineage_cache):,} unique taxids')

lin_df = pd.DataFrame(lineages)
df = pd.concat([df.reset_index(drop=True), lin_df.reset_index(drop=True)], axis=1)

for col in ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']:
    if col not in df.columns:
        df[col] = 'Unknown'
    df[col] = df[col].fillna('Unknown')

print(f'\nDataFrame columns: {list(df.columns)}')
known_species = (df['species'] != 'Unknown').sum()
print(f'Species coverage: {known_species:,} / {len(df):,} ({known_species/len(df)*100:.1f}%)')
print(f'Unique species: {df[df["species"] != "Unknown"]["species"].nunique():,}')
print()
print(df[['accession', 'taxid', 'kingdom', 'genus', 'species']].head(10).to_string())
print('\n✅ Taxonomy resolution complete.')

---
## 7 · Data Cleaning

In [ ]:
print(f'Starting size: {len(df):,}')

# 1) Drop Unknown species
df = df[df['species'] != 'Unknown'].copy()
print(f'After dropping Unknown species: {len(df):,}')
assert len(df) > 0, 'No sequences with known species!'

# 2) Valid DNA characters only (A, C, G, T, possibly N, R, Y, etc.)
df['seq_upper'] = df['sequence'].str.upper().str.strip()
valid_dna = df['seq_upper'].str.match(r'^[ACGTNRYSWKMBDHV]+$')
n_invalid = (~valid_dna).sum()
df = df[valid_dna].copy()
print(f'After valid-DNA filter: {len(df):,} (dropped {n_invalid:,})')
assert len(df) > 0, 'No sequences with valid DNA!'

# 3) Replace ambiguous bases with N for cleaner input
df['seq_clean'] = df['seq_upper'].str.replace(r'[^ACGT]', 'N', regex=True)

# 4) Length filter
df['seq_len'] = df['seq_clean'].str.len()
before_len = len(df)
df = df[(df['seq_len'] >= MIN_SEQ_LEN) & (df['seq_len'] <= MAX_SEQ_LEN)].copy()
print(f'After length filter ({MIN_SEQ_LEN}-{MAX_SEQ_LEN} nt): {len(df):,} (dropped {before_len - len(df):,})')
assert len(df) > 0, 'No sequences in length range!'

# 5) Drop exact duplicate sequences (keep first)
before_dedup = len(df)
df = df.drop_duplicates(subset='seq_clean', keep='first').copy()
print(f'After deduplication: {len(df):,}  (removed {before_dedup - len(df):,})')

print(f'\nSequence length stats:')
print(df['seq_len'].describe().to_string())
print(f'\nUnique species: {df["species"].nunique():,}')
print(f'Total sequences: {len(df):,}')
print('\n✅ Data cleaning complete.')

---
## 8 · Class Balancing Strategy

Goal: avoid degenerate one-sample-per-class while keeping diverse coverage.

In [ ]:
print(f'=== Class Balancing ===')
print(f'  MIN_SAMPLES_PER_SPECIES = {MIN_SAMPLES_PER_SPECIES}')
print(f'  MAX_CLASSES             = {MAX_CLASSES}')
print(f'  MAX_PER_CLASS           = {MAX_PER_CLASS}')
print(f'  TARGET_TOTAL            = {TARGET_TOTAL:,}')

species_counts = df['species'].value_counts()
print(f'\nBefore filtering: {len(species_counts):,} species')
print(f'  Top-5 species by count:')
for sp, cnt in species_counts.head().items():
    print(f'    {sp}: {cnt}')
print(f'  Bottom-5 species by count:')
for sp, cnt in species_counts.tail().items():
    print(f'    {sp}: {cnt}')

# Distribution stats
print(f'\n  Species with 1 sample:     {(species_counts == 1).sum():,}')
print(f'  Species with 2-4 samples:  {((species_counts >= 2) & (species_counts <= 4)).sum():,}')
print(f'  Species with 5-9 samples:  {((species_counts >= 5) & (species_counts <= 9)).sum():,}')
print(f'  Species with 10+ samples:  {(species_counts >= 10).sum():,}')
print(f'  Species with 50+ samples:  {(species_counts >= 50).sum():,}')

# 1) Enforce minimum samples
valid_species = species_counts[species_counts >= MIN_SAMPLES_PER_SPECIES].index
print(f'\nSpecies with >= {MIN_SAMPLES_PER_SPECIES} samples: {len(valid_species):,}')

# If too few species, try lowering threshold
if len(valid_species) < 20:
    for threshold in [4, 3, 2]:
        fallback = species_counts[species_counts >= threshold].index
        if len(fallback) >= 20:
            print(f'  ⚠ Only {len(valid_species)} species at threshold {MIN_SAMPLES_PER_SPECIES}, '
                  f'falling back to {threshold} → {len(fallback)} species')
            valid_species = fallback
            MIN_SAMPLES_PER_SPECIES = threshold
            break

assert len(valid_species) >= 10, f'Only {len(valid_species)} species meet threshold — data is insufficient!'

# 2) Keep top N most frequent
if len(valid_species) > MAX_CLASSES:
    valid_species = species_counts.loc[valid_species].nlargest(MAX_CLASSES).index
    print(f'  Capped to top {MAX_CLASSES}: {len(valid_species):,} species')

df_bal = df[df['species'].isin(valid_species)].copy()
print(f'  Sequences in selected species: {len(df_bal):,}')

# 3) Cap per-class
df_bal = df_bal.groupby('species', group_keys=False).apply(
    lambda g: g.sample(n=min(len(g), MAX_PER_CLASS), random_state=42)
).reset_index(drop=True)
print(f'  After per-class cap ({MAX_PER_CLASS}): {len(df_bal):,}')

# 4) If still too large, subsample to target
if len(df_bal) > TARGET_TOTAL:
    # Subsample while maintaining species distribution
    frac = TARGET_TOTAL / len(df_bal)
    df_bal = df_bal.groupby('species', group_keys=False).apply(
        lambda g: g.sample(n=max(MIN_SAMPLES_PER_SPECIES, int(len(g) * frac)), random_state=42)
    ).reset_index(drop=True)
    print(f'  Subsampled towards target: {len(df_bal):,}')

final_counts = df_bal['species'].value_counts()
print(f'\n=== Final Dataset ===')
print(f'  Sequences : {len(df_bal):,}')
print(f'  Species   : {df_bal["species"].nunique():,}')
print(f'  Min/class : {final_counts.min()}')
print(f'  Max/class : {final_counts.max()}')
print(f'  Mean/class: {final_counts.mean():.1f}')
print(f'  Median/class : {final_counts.median():.0f}')
print('\n✅ Balancing complete.')

# Replace main df
df = df_bal

---
## 9 · Stratified Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# For stratified split to work, each class needs >= 3 samples (for 80/10/10)
# Filter out any class with < 3 samples to avoid split failures
species_counts_final = df['species'].value_counts()
too_few = species_counts_final[species_counts_final < 3].index
if len(too_few) > 0:
    print(f'Removing {len(too_few)} species with < 3 samples (can\'t stratify)')
    df = df[~df['species'].isin(too_few)].copy()
    print(f'  After removal: {len(df):,} sequences, {df["species"].nunique():,} species')

# 80 / 10 / 10
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['species']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['species']
)

print(f'Train : {len(train_df):,}  ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val   : {len(val_df):,}  ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test  : {len(test_df):,}  ({len(test_df)/len(df)*100:.1f}%)')

# Sanity checks
train_species = set(train_df['species'])
val_species   = set(val_df['species'])
test_species  = set(test_df['species'])

val_only  = val_species - train_species
test_only = test_species - train_species
print(f'\nSpecies in val but not train : {len(val_only)}')
print(f'Species in test but not train: {len(test_only)}')

# If any species only in val/test, move those samples to train
if val_only:
    move_mask = val_df['species'].isin(val_only)
    train_df = pd.concat([train_df, val_df[move_mask]])
    val_df = val_df[~move_mask]
    print(f'  Moved {move_mask.sum()} val samples to train.')

if test_only:
    move_mask = test_df['species'].isin(test_only)
    train_df = pd.concat([train_df, test_df[move_mask]])
    test_df = test_df[~move_mask]
    print(f'  Moved {move_mask.sum()} test samples to train.')

# Final assertion
assert len(train_df) > 0, 'Train set is empty!'
assert len(val_df) > 0, 'Val set is empty!'
assert len(test_df) > 0, 'Test set is empty!'
assert set(val_df['species']).issubset(set(train_df['species'])), 'Val has unseen labels!'
assert set(test_df['species']).issubset(set(train_df['species'])), 'Test has unseen labels!'

print(f'\nFinal splits: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
print(f'Train species: {train_df["species"].nunique():,}')
print('✅ Splits OK — no unseen labels in val/test.')

---
## 10 · Label Encoding (from train split)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle, numpy as np

le = LabelEncoder()
le.fit(train_df['species'])

num_classes = len(le.classes_)
print(f'Number of classes: {num_classes}')

train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

train_df['label'] = le.transform(train_df['species'])
val_df['label']   = le.transform(val_df['species'])
test_df['label']  = le.transform(test_df['species'])

# Persist label encoder
le_path = MODEL_DIR / 'label_encoder.pkl'
with open(le_path, 'wb') as f:
    pickle.dump(le, f)
print(f'Saved label encoder to {le_path}')

# Compute class weights for loss
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    'balanced', classes=np.arange(num_classes), y=train_df['label'].values
)
class_weights = class_weights / class_weights.mean()  # normalize around 1
imbalance_ratio = class_weights.max() / class_weights.min()
print(f'Class weight range: [{class_weights.min():.3f}, {class_weights.max():.3f}]')
print(f'Imbalance ratio: {imbalance_ratio:.1f}x')
print('✅ Label encoding complete.')

---
## 11 · Select & Instantiate DNA Model

Strategy:
1. Try a standard BERT-based DNA model that works natively with `AutoModelForSequenceClassification`
2. Fall back through candidates if one fails
3. Verify save/reload compatibility

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, BertTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# ── Model candidates (ordered by preference) ──────────────────────
# We prioritize models that:
# 1. Work with standard AutoModelForSequenceClassification (no custom remote code)
# 2. Are compatible with P100 GPU (no exotic CUDA kernels)
# 3. Have reasonable size for fine-tuning on 16GB VRAM
MODEL_CANDIDATES = [
    # InstaDeep Nucleotide Transformer V2 (ESM-based, standard architecture)
    'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species',
    # Larger variant
    'InstaDeepAI/nucleotide-transformer-50m-multi-species',
    # DNABERT-2 (needs trust_remote_code but is excellent)
    'zhihan1996/DNABERT-2-117M',
]

selected_model = None
tokenizer = None
model = None
needs_trust_remote_code = False

for candidate in MODEL_CANDIDATES:
    print(f'\n{"="*60}')
    print(f'Trying: {candidate}')
    print(f'{"="*60}')
    try:
        # Step 1: Load tokenizer
        print('  Loading tokenizer ...')
        tok = AutoTokenizer.from_pretrained(candidate, trust_remote_code=True)
        print(f'  ✓ Tokenizer loaded (vocab_size={tok.vocab_size})')
        
        # Step 2: Load config + set num_labels
        print('  Loading config ...')
        cfg = AutoConfig.from_pretrained(candidate, trust_remote_code=True)
        cfg.num_labels = num_classes
        print(f'  ✓ Config loaded (hidden_size={getattr(cfg, "hidden_size", "?")})')
        
        # Step 3: Load model for sequence classification
        print('  Loading model ...')
        mdl = AutoModelForSequenceClassification.from_pretrained(
            candidate,
            config=cfg,
            trust_remote_code=True,
            ignore_mismatched_sizes=True
        )
        print(f'  ✓ Model loaded ({sum(p.numel() for p in mdl.parameters())/1e6:.1f}M params)')
        
        # Step 4: Test GPU forward pass to catch CUDA kernel issues early
        print('  Testing GPU forward pass ...')
        mdl_test = mdl.to(device)
        test_input = tok('ACGTACGTACGT', return_tensors='pt', padding='max_length', 
                        max_length=32, truncation=True)
        test_input = {k: v.to(device) for k, v in test_input.items()}
        with torch.no_grad():
            test_out = mdl_test(**test_input)
        assert test_out.logits.shape[-1] == num_classes, \
            f'Output classes {test_out.logits.shape[-1]} != expected {num_classes}'
        print(f'  ✓ GPU forward pass OK (output shape: {test_out.logits.shape})')
        mdl_test = mdl_test.cpu()
        del test_input, test_out
        torch.cuda.empty_cache()
        
        # Step 5: Test save/reload
        print('  Testing save/reload ...')
        test_save_dir = WORK_DIR / '_model_test'
        test_save_dir.mkdir(exist_ok=True)
        mdl.save_pretrained(test_save_dir)
        tok.save_pretrained(test_save_dir)
        
        # Try loading WITHOUT trust_remote_code first
        try:
            _test_mdl = AutoModelForSequenceClassification.from_pretrained(str(test_save_dir))
            _test_tok = AutoTokenizer.from_pretrained(str(test_save_dir))
            del _test_mdl, _test_tok
            print(f'  ✓ Standard reload OK (no trust_remote_code needed)')
            needs_trust_remote_code = False
        except Exception as e1:
            print(f'  ⚠ Standard reload failed: {type(e1).__name__}: {str(e1)[:100]}')
            _test_mdl = AutoModelForSequenceClassification.from_pretrained(
                str(test_save_dir), trust_remote_code=True
            )
            _test_tok = AutoTokenizer.from_pretrained(
                str(test_save_dir), trust_remote_code=True
            )
            del _test_mdl, _test_tok
            print(f'  ✓ Reload OK with trust_remote_code=True')
            needs_trust_remote_code = True
        
        shutil.rmtree(test_save_dir)
        selected_model = candidate
        tokenizer = tok
        model = mdl
        print(f'\n✅ Selected model: {selected_model}')
        if needs_trust_remote_code:
            print('   Note: requires trust_remote_code=True for loading')
        break
        
    except Exception as e:
        print(f'  ✗ Failed: {type(e).__name__}: {e}')
        import traceback
        traceback.print_exc()
        # Clean up any partial state
        torch.cuda.empty_cache()
        continue

assert model is not None, 'All model candidates failed to load! Check errors above.'

# Move to GPU
model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParameters: {total_params/1e6:.1f}M total, {trainable/1e6:.1f}M trainable')
print(f'Num classes: {model.config.num_labels}')

---
## 12 · Tokenization + Dataset

In [ ]:
from torch.utils.data import Dataset as TorchDataset

class DNADataset(TorchDataset):
    """Pre-tokenized DNA sequence dataset for efficient training."""
    def __init__(self, sequences, labels, tokenizer, max_length):
        self.labels = labels
        # Pre-tokenize all sequences for efficiency
        print(f'    Tokenizing {len(sequences):,} sequences (max_length={max_length}) ...')
        self.encodings = tokenizer(
            sequences,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        print(f'    Done. input_ids shape: {self.encodings["input_ids"].shape}')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }
        # Include token_type_ids if present (some models need it)
        if 'token_type_ids' in self.encodings:
            item['token_type_ids'] = self.encodings['token_type_ids'][idx]
        return item

print('Building datasets ...')
train_dataset = DNADataset(
    train_df['seq_clean'].tolist(), train_df['label'].tolist(),
    tokenizer, MAX_TOKEN_LEN
)
val_dataset = DNADataset(
    val_df['seq_clean'].tolist(), val_df['label'].tolist(),
    tokenizer, MAX_TOKEN_LEN
)
test_dataset = DNADataset(
    test_df['seq_clean'].tolist(), test_df['label'].tolist(),
    tokenizer, MAX_TOKEN_LEN
)

print(f'\n  Train: {len(train_dataset):,}')
print(f'  Val  : {len(val_dataset):,}')
print(f'  Test : {len(test_dataset):,}')

# Quick sanity check
sample = train_dataset[0]
print(f'\nSample input_ids shape: {sample["input_ids"].shape}')
print(f'Sample attention_mask shape: {sample["attention_mask"].shape}')
print(f'Sample label: {sample["labels"]}')
print(f'Non-padding tokens: {sample["attention_mask"].sum().item()}')
print('✅ Datasets ready.')

---
## 13 · Fine-Tune with HuggingFace Trainer

In [ ]:
import numpy as np
from transformers import Trainer, TrainingArguments
from torch import nn

# ── Custom Trainer with optional class-weighted loss ───────────────
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self._class_weights = class_weights  # store raw, move to device in compute_loss

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        if self._class_weights is not None:
            weight = torch.tensor(
                self._class_weights, dtype=torch.float32, device=logits.device
            )
            loss_fn = nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fn = nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Metrics ────────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    # top-3 accuracy
    top3 = np.argsort(logits, axis=-1)[:, -3:]
    top3_acc = np.mean([l in t for l, t in zip(labels, top3)])
    return {'accuracy': acc, 'macro_f1': f1, 'top3_accuracy': top3_acc}

# ── Compute actual warmup steps ───────────────────────────────────
steps_per_epoch = max(1, len(train_dataset) // TRAIN_BATCH)
total_steps = steps_per_epoch * EPOCHS
warmup = min(WARMUP_STEPS, total_steps // 5)  # cap at 20% of total
print(f'Steps per epoch: {steps_per_epoch}')
print(f'Total steps: {total_steps}')
print(f'Warmup steps: {warmup}')

# ── Training arguments ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=str(WORK_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup,
    fp16=FP16 and torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
    dataloader_num_workers=2,
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    group_by_length=False,
)

# Use class weights only if imbalance ratio > 5x
use_weights = imbalance_ratio > 5.0
print(f'\nImbalance ratio: {imbalance_ratio:.1f}x → {"using" if use_weights else "skipping"} class weights')

trainer = WeightedTrainer(
    class_weights=class_weights if use_weights else None,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(f'\n🚀 Starting training: {EPOCHS} epochs, {len(train_dataset):,} samples')
print(f'   Batch size: {TRAIN_BATCH},  LR: {LEARNING_RATE},  FP16: {training_args.fp16}')
print(f'   Warmup steps: {warmup}')
print(f'   Num classes: {num_classes}')
train_result = trainer.train()

print('\n=== Training Complete ===')
print(f'  Train loss:  {train_result.training_loss:.4f}')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v}')

---
## 14 · Test Evaluation

In [ ]:
print('=== Test Set Evaluation ===')
test_metrics = trainer.evaluate(test_dataset)

for k, v in sorted(test_metrics.items()):
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

test_accuracy  = test_metrics.get('eval_accuracy', 0)
test_f1        = test_metrics.get('eval_macro_f1', 0)
test_top3      = test_metrics.get('eval_top3_accuracy', 0)

print(f'\n📊 Test Accuracy:      {test_accuracy:.4f}')
print(f'📊 Test Macro-F1:      {test_f1:.4f}')
print(f'📊 Test Top-3 Accuracy: {test_top3:.4f}')

if test_accuracy < 0.01:
    print('\n⚠️  WARNING: Test accuracy is extremely low. Model may not have converged.')
    print('    Consider: more epochs, higher learning rate, or check data quality.')

print('\n✅ Evaluation complete.')

---
## 15 · Save Plug-and-Play Model Artifacts

In [ ]:
import json

print(f'Saving model to {MODEL_DIR} ...')

# Save model + tokenizer
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print('  ✓ Model weights + config saved')
print('  ✓ Tokenizer saved')

# label_encoder.pkl already saved earlier — verify
assert (MODEL_DIR / 'label_encoder.pkl').exists(), 'label_encoder.pkl missing!'
print('  ✓ label_encoder.pkl present')

# model_info.json
model_info = {
    'base_model': selected_model,
    'trust_remote_code': needs_trust_remote_code,
    'num_classes': num_classes,
    'max_length': MAX_TOKEN_LEN,
    'species_list': le.classes_.tolist(),
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
    'metrics': {
        'test_accuracy': round(test_accuracy, 4),
        'test_macro_f1': round(test_f1, 4),
        'test_top3_accuracy': round(test_top3, 4),
    },
    'preprocessing': {
        'min_seq_len': MIN_SEQ_LEN,
        'max_seq_len': MAX_SEQ_LEN,
        'valid_dna_regex': r'^[ACGTNRYSWKMBDHV]+$',
        'ambiguous_base_replacement': 'N',
        'min_samples_per_species': MIN_SAMPLES_PER_SPECIES,
        'max_classes': MAX_CLASSES,
        'max_per_class': MAX_PER_CLASS,
    },
    'training': {
        'epochs': EPOCHS,
        'batch_size': TRAIN_BATCH,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_steps': warmup,
        'fp16': FP16,
        'class_weighted_loss': use_weights,
    },
    'target_project_dir': 'edna_pipeline/models/dnabert2_16s_species',
}

with open(MODEL_DIR / 'model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print('  ✓ model_info.json saved')

# List all saved files
print(f'\nSaved artifacts:')
for p in sorted(MODEL_DIR.iterdir()):
    size = p.stat().st_size
    unit = 'KB' if size < 1e6 else 'MB'
    val  = size / 1e3 if size < 1e6 else size / 1e6
    print(f'  {p.name:45s} {val:8.1f} {unit}')

print('\n✅ All artifacts saved.')

---
## 16 · Compatibility Check (Mandatory)

Reload the saved model from disk and run inference to confirm plug-and-play.

In [ ]:
import torch, pickle, json
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print('=== Compatibility Check ===')
print(f'Loading model from: {MODEL_DIR}')

# Read model_info to check if trust_remote_code is needed
with open(MODEL_DIR / 'model_info.json') as f:
    saved_info = json.load(f)
trc = saved_info.get('trust_remote_code', False)

# 1) Load model
try:
    check_model = AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR), trust_remote_code=trc
    )
    print(f'  ✓ Model loaded (trust_remote_code={trc})')
except Exception as e:
    print(f'  First attempt failed: {e}')
    check_model = AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR), trust_remote_code=True
    )
    print('  ✓ Model loaded (fallback trust_remote_code=True)')

check_model.eval()
check_device = 'cuda' if torch.cuda.is_available() else 'cpu'
check_model = check_model.to(check_device)

# 2) Load tokenizer
try:
    check_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=trc)
    print(f'  ✓ Tokenizer loaded (trust_remote_code={trc})')
except Exception:
    check_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
    print('  ✓ Tokenizer loaded (fallback trust_remote_code=True)')

# 3) Load label encoder
with open(MODEL_DIR / 'label_encoder.pkl', 'rb') as f:
    check_le = pickle.load(f)
print(f'  ✓ Label encoder loaded ({len(check_le.classes_)} classes)')

# 4) Run inference on a sample sequence
sample_seq = test_df.iloc[0]['seq_clean']
true_species = test_df.iloc[0]['species']

inputs = check_tokenizer(
    sample_seq, return_tensors='pt', truncation=True,
    padding='max_length', max_length=MAX_TOKEN_LEN
)
inputs = {k: v.to(check_device) for k, v in inputs.items()}

with torch.no_grad():
    logits = check_model(**inputs).logits

pred_label = logits.argmax(dim=-1).item()
pred_species = check_le.inverse_transform([pred_label])[0]
confidence = torch.softmax(logits, dim=-1).max().item()

print(f'\n  Sample sequence: {sample_seq[:80]}...')
print(f'  True species:    {true_species}')
print(f'  Predicted:       {pred_species}')
print(f'  Confidence:      {confidence:.4f}')
print(f'  Correct:         {"✓" if pred_species == true_species else "✗"}')

# Top-3 predictions
top3_idx = logits.argsort(dim=-1, descending=True)[0, :3].cpu().numpy()
top3_probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
print(f'\n  Top-3 predictions:')
for i, idx in enumerate(top3_idx):
    sp = check_le.inverse_transform([idx])[0]
    print(f'    {i+1}. {sp}  ({top3_probs[idx]:.4f})')

del check_model, check_tokenizer  # free memory
torch.cuda.empty_cache()

print('\n✅ Compatibility check PASSED — model is plug-and-play.')

---
## 17 · Create ZIP Artifact

In [ ]:
import shutil, zipfile

ZIP_PATH = OUTPUT_DIR / 'dnabert2_16s_model.zip'

# Remove old zip if exists
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Create zip (the archive will contain a folder named dnabert2_16s_species)
zip_base = shutil.make_archive(
    base_name=str(ZIP_PATH).replace('.zip', ''),
    format='zip',
    root_dir=str(OUTPUT_DIR),
    base_dir='dnabert2_16s_species'
)

zip_size_mb = pathlib.Path(zip_base).stat().st_size / 1e6
print(f'\n📦 ZIP created: {zip_base}')
print(f'   Size: {zip_size_mb:.1f} MB')

# Verify zip contents
with zipfile.ZipFile(zip_base, 'r') as zf:
    names = zf.namelist()
    print(f'   Files in zip: {len(names)}')
    for n in sorted(names):
        info = zf.getinfo(n)
        print(f'     {n:55s}  {info.file_size/1e3:8.1f} KB')

# Verify critical files are in the zip
required_files = ['config.json', 'label_encoder.pkl', 'model_info.json']
for req in required_files:
    matches = [n for n in names if n.endswith(req)]
    assert len(matches) > 0, f'Missing {req} in zip!'

# Check for model weights
has_weights = any(n.endswith('.safetensors') or n.endswith('.bin') for n in names)
assert has_weights, 'No model weight files found in zip!'

print('\n✅ ZIP artifact ready and verified.')

---
## 18 · Download Instructions

### How to download the model:

**Option A — Kaggle/Colab file browser (simplest):**
1. In the left sidebar, click the **📁 Files** icon.
2. Navigate to: `output/dnabert2_16s_model/`
3. Right-click `dnabert2_16s_model.zip` → **Download**

**Option B — Kaggle: Save as output dataset:**
The zip is saved at `/content/output/dnabert2_16s_model/dnabert2_16s_model.zip`.
On Kaggle, notebook outputs are automatically saved. Download from the notebook's Output tab.

**Option C — Colab: Run this cell to trigger browser download:**
```python
from google.colab import files
files.download('/content/output/dnabert2_16s_model/dnabert2_16s_model.zip')
```

### After downloading, deploy to your project:

```bash
# On your local machine:
cd /home/megh/edna/edna_pipeline/models/
unzip ~/Downloads/dnabert2_16s_model.zip
# This creates: dnabert2_16s_species/ with all artifacts
```

### Expected directory structure after unzip:
```
/home/megh/edna/edna_pipeline/models/dnabert2_16s_species/
├── config.json
├── model.safetensors  (or pytorch_model.bin)
├── tokenizer_config.json
├── tokenizer.json  (or vocab.txt + special_tokens_map.json)
├── special_tokens_map.json
├── label_encoder.pkl
└── model_info.json
```

### Quick local verification:
```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pickle, json

model_dir = '/home/megh/edna/edna_pipeline/models/dnabert2_16s_species'
with open(f'{model_dir}/model_info.json') as f:
    info = json.load(f)
trc = info.get('trust_remote_code', False)
model = AutoModelForSequenceClassification.from_pretrained(model_dir, trust_remote_code=trc)
tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=trc)
with open(f'{model_dir}/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
print(f"Model: {info['base_model']}, {info['num_classes']} species")
print(f"Test accuracy: {info['metrics']['test_accuracy']}")
```

In [ ]:
print('='*60)
print('  🎉  NOTEBOOK COMPLETE  🎉')
print('='*60)
print()
print(f'  Model:     {selected_model}')
print(f'  Classes:   {num_classes} species')
print(f'  Train:     {len(train_df):,} sequences')
print(f'  Val:       {len(val_df):,} sequences')
print(f'  Test:      {len(test_df):,} sequences')
print(f'  Accuracy:  {test_accuracy:.4f}')
print(f'  Macro-F1:  {test_f1:.4f}')
print(f'  Top-3 Acc: {test_top3:.4f}')
print()
print(f'  ZIP: {ZIP_PATH}')
print(f'  Size: {zip_size_mb:.1f} MB')
print()
print('  Deploy with:')
print('    cd /home/megh/edna/edna_pipeline/models/')
print('    unzip dnabert2_16s_model.zip')
print()
print('='*60)